# Session 0 — Your turn: From text to probabilities

These tasks use GPT-2's tokenizer (it downloads once, a few hundred kilobytes) and plain arithmetic. No GPU is needed.

This is the **solutions** notebook: every task is filled in and every check passes. Try the tasks yourself first.

In [1]:
import sys; sys.path[:0] = ["..", "."]   # aij_check.py: in the folder above, or beside this notebook
import numpy as np
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("gpt2")
from aij_check import check, close, summary

## Task 1: How many tokens, and what do they cost?

Fill `n_tokens` (a dictionary string -> number of GPT-2 tokens) for the four strings below.
Then set `chars_per_token` for the English and the Arabic sentence (characters divided by tokens, with `len(s)` counting characters) and `arabic_costs_more` to True if the Arabic sentence needs more tokens than the English one.

In [2]:
strings = {"space": " battery", "no space": "battery", "english": "The battery is empty, so the robot stopped working.", "arabic": "البطارية فارغة، لذلك توقف الروبوت عن العمل."}
n_tokens = {name: len(tok(s)["input_ids"]) for name, s in strings.items()}
chars_per_token = {k: len(strings[k]) / n_tokens[k] for k in ("english", "arabic")}
arabic_costs_more = n_tokens["arabic"] > n_tokens["english"]
print(n_tokens, chars_per_token)

{'space': 1, 'no space': 2, 'english': 11, 'arabic': 45} {'english': 4.636363636363637, 'arabic': 0.9555555555555556}


In [3]:
check("' battery' with a space is one token", lambda: n_tokens["space"] == 1)
check("'battery' without a space is two tokens", lambda: n_tokens["no space"] == 2, "The space belongs to the word in GPT-2's vocabulary.")
check("the English sentence takes 11 tokens", lambda: n_tokens["english"] == 11)
check("Arabic gets less than one character per token", lambda: chars_per_token["arabic"] < 1.0)
check("Arabic costs more tokens", lambda: arabic_costs_more is True)
summary()

PASS  ' battery' with a space is one token
PASS  'battery' without a space is two tokens
PASS  the English sentence takes 11 tokens
PASS  Arabic gets less than one character per token
PASS  Arabic costs more tokens

5 of 5 checks passed.  Well done.


**Why it works.** GPT-2's vocabulary was learned mostly from English web text, so English words are single tokens while Arabic is split down to bytes. The same message costs several times more tokens, which means more compute and less context.

## Task 2: Bits, perplexity and the tokenizer

A model gave these probabilities to the eight tokens of a text: `probs`. Compute `bits` (the bits each token cost, `-log2 p`), `mean_bits`, `perplexity` (`2 ** mean_bits`) and `bits_per_character`, given that the text has 31 characters.
A second model with a different tokenizer needed 4 tokens for the same text and paid 4.0 bits per token: store its `bits_per_character_b` too, and `b_is_better` (lower bits per character is better).

In [4]:
probs = np.array([0.30, 0.05, 0.60, 0.02, 0.25, 0.10, 0.45, 0.80])
bits = -np.log2(probs)
mean_bits = bits.mean()
perplexity = 2 ** mean_bits
bits_per_character = bits.sum() / 31
bits_per_character_b = 4 * 4.0 / 31
b_is_better = bool(bits_per_character_b < bits_per_character)
print(mean_bits, perplexity, bits_per_character, bits_per_character_b)

2.404446844526784 5.294325311921921 0.620502411490783 0.5161290322580645


In [5]:
check("the rarest token (p = 0.02) costs about 5.64 bits", lambda: close(bits[3], 5.644, 0.001))
check("mean bits per token", lambda: close(mean_bits, 2.4044, 0.01))
check("perplexity is 2 to the mean bits", lambda: close(perplexity, 2 ** mean_bits, 1e-9))
check("bits per character uses the whole text", lambda: close(bits_per_character, bits.sum() / 31, 1e-9))
check("model B has the HIGHER perplexity per token but the LOWER cost per character", lambda: b_is_better is True and 2 ** 4.0 > perplexity)
summary()

PASS  the rarest token (p = 0.02) costs about 5.64 bits
PASS  mean bits per token
PASS  perplexity is 2 to the mean bits
PASS  bits per character uses the whole text
PASS  model B has the HIGHER perplexity per token but the LOWER cost per character

5 of 5 checks passed.  Well done.


**Why it works.** Perplexity per token depends on how the text was cut into tokens: a tokenizer with fewer, bigger tokens pays more bits per token but fewer in total. Only a per-character (or per-byte) number compares models with different tokenizers.